# Tome 2, Chapter 9: Finite Fields (Python companion)

Exercises first, solutions at the end. We build $\mathbb F_{256}$ from scratch, the field of the AES cipher and of QR codes, compute the AES S-box from field inverses, count irreducible polynomials, and then correct errors: a Hamming code that fixes one flipped bit, and a Reed–Solomon code over $\mathbb F_{256}$ decoded with the Berlekamp–Welch algorithm of Theorem 9.12.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import random
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from collections import Counter
import sympy
from sympy import Poly, symbols, mobius, divisors
X = symbols('X')
random.seed(2026)

## Exercises

**P9.1 (★) $\mathbb F_{256}$ from scratch.** Represent an element of $\mathbb F_2[X]/(X^8+X^4+X^3+X+1)$ by a byte (bit $i$ is the coefficient of $X^i$). Addition is XOR. Write multiplication (shift and reduce), check the example of the AES standard $\{57\}\cdot\{83\}=\{c1\}$, check that $\{03\}$ is a primitive element while $\{02\}$ is not (what is its order?), and build log/antilog tables to compute inverses.

In [ ]:
# Your code here


**P9.2 (★) The AES S-box.** The S-box of AES sends a byte $b$ to $A\cdot b^{-1}+\{63\}$, where $b^{-1}$ is the inverse in $\mathbb F_{256}$ (with $0^{-1}=0$) and $A$ is the $\mathbb F_2$-linear map $b\mapsto b\oplus\mathrm{rot}_1(b)\oplus\mathrm{rot}_2(b)\oplus\mathrm{rot}_3(b)\oplus\mathrm{rot}_4(b)$ (bit rotations to the left). Compute it, check $S(\{00\})=\{63\}$ and $S(\{53\})=\{ed\}$, and check that $S$ is a permutation without fixed points. Why use the field inverse? Measure the *differential uniformity* of $x\mapsto x^{-1}$: the maximum over $a\ne0$, $b$ of the number of $x$ with $x^{-1}\oplus(x\oplus a)^{-1}=b$.

In [ ]:
# Your code here


**P9.3 (★) Counting irreducibles.** Over $\mathbb F_2$ (degrees $\le 10$) and $\mathbb F_3$ (degrees $\le 6$), count the monic irreducible polynomials by a sieve (cross out all products), and compare with Theorem 9.10. Then factor $X^{9}-X$ and $X^{27}-X$ over $\mathbb F_3$ and check Theorem 9.9.

In [ ]:
# Your code here


**P9.4 (★★) The Hamming code.** The $[7,4]$ Hamming code sends $4$ data bits as $7$ bits $c$ with $Hc=0$ over $\mathbb F_2$, where the columns of the $3\times7$ matrix $H$ are the binary expansions of $1,\dots,7$. Show by computation that flipping any single bit of a codeword is detected and located by the *syndrome* $Hy$, and that two flips are always detected but wrongly corrected.

In [ ]:
# Your code here


**P9.5 (★★★) Reed–Solomon over $\mathbb F_{256}$.** Encode a $20$-byte message as the values of a polynomial of degree $<20$ at $30$ points, corrupt $5$ random bytes, and decode with Berlekamp–Welch (Theorem 9.12) by Gaussian elimination over $\mathbb F_{256}$. What happens with $6$ errors? Finally, solve Exercise 9.7 over $\mathbb F_7$ with the same code.

In [ ]:
# Your code here


---
## Solutions

### P9.1

In [ ]:
AES_POLY = 0x11B          # X^8 + X^4 + X^3 + X + 1

def gmul(a, b, poly=AES_POLY):
    r = 0
    while b:
        if b & 1:
            r ^= a
        a <<= 1
        if a & 0x100:
            a ^= poly        # reduce: X^8 = X^4 + X^3 + X + 1
        b >>= 1
    return r

print('57 * 83 =', hex(gmul(0x57, 0x83)))

def order(g):
    x, k = g, 1
    while x != 1:
        x, k = gmul(x, g), k + 1
    return k
print('order of 02:', order(0x02), '  order of 03:', order(0x03))

EXP, LOG = [0] * 510, [0] * 256
x = 1
for i in range(255):
    EXP[i] = EXP[i + 255] = x
    LOG[x] = i
    x = gmul(x, 0x03)
def ginv(a):
    return 0 if a == 0 else EXP[255 - LOG[a]]
def gmul_fast(a, b):
    return 0 if a == 0 or b == 0 else EXP[LOG[a] + LOG[b]]

assert all(gmul(a, ginv(a)) == 1 for a in range(1, 256))
assert all(gmul(a, b) == gmul_fast(a, b) for a in range(256) for b in range(256))
# a few field axioms on random triples
for _ in range(2000):
    a, b, c = (random.randrange(256) for _ in range(3))
    assert gmul(a, b ^ c) == gmul(a, b) ^ gmul(a, c) and gmul(gmul(a, b), c) == gmul(a, gmul(b, c))
print('inverse of 53:', hex(ginv(0x53)), '  all checks passed')

$\{57\}\cdot\{83\}=\{c1\}$, as in the AES standard (FIPS 197). The element $X=\{02\}$ has order $51$, a proper divisor of $255=3\cdot5\cdot17$: so $X^8+X^4+X^3+X+1$ is irreducible but *not* primitive (Exercise 9.3), and the log tables use $\{03\}=X+1$, which has order $255$. Log tables turn each product into two lookups and an addition, which is how fast software implementations of Reed–Solomon codes multiply.

### P9.2

In [ ]:
def rotl(b, k):
    return ((b << k) | (b >> (8 - k))) & 0xFF
def sbox(b):
    v = ginv(b)
    return v ^ rotl(v, 1) ^ rotl(v, 2) ^ rotl(v, 3) ^ rotl(v, 4) ^ 0x63
S = [sbox(b) for b in range(256)]
print('S(00) =', hex(S[0]), ' S(53) =', hex(S[0x53]))
print('permutation:', sorted(S) == list(range(256)), '  fixed points:', [b for b in range(256) if S[b] == b])
print('first row:', ' '.join(f'{v:02x}' for v in S[:16]))

def diff_uniformity(f):
    return max(max(Counter(f(x) ^ f(x ^ a) for x in range(256)).values()) for a in range(1, 256))
print('differential uniformity of the inverse:', diff_uniformity(ginv))
print('differential uniformity of a random permutation:', diff_uniformity(lambda x, P=random.sample(range(256), 256): P[x]))

The first row `63 7c 77 7b f2 6b 6f c5 30 01 67 2b fe d7 ab 76` is the one printed in every AES reference. The inverse map has differential uniformity $4$: for every input difference $a\ne0$, each output difference occurs for at most $4$ of the $256$ inputs, which is close to the theoretical optimum $2$ and much better than a random permutation (typically $10$ or so). This resistance to *differential cryptanalysis*, provable because $x^{-1}$ is a field operation, is why the designers built the S-box from the inverse in $\mathbb F_{256}$; the affine map $A$ breaks its algebraic simplicity ($0\mapsto0$, $1\mapsto1$).

### P9.3

In [ ]:
def sieve_counts(p, nmax):
    # all monic polynomials of degree <= nmax, as tuples of coefficients (constant first)
    def monic(n):
        for c in itertools.product(range(p), repeat=n):
            yield c + (1,)
    def mul(a, b):
        r = [0] * (len(a) + len(b) - 1)
        for i, x in enumerate(a):
            if x:
                for j, y in enumerate(b):
                    r[i + j] = (r[i + j] + x * y) % p
        return tuple(r)
    irr = {n: [] for n in range(1, nmax + 1)}
    for n in range(1, nmax + 1):
        composite = set()
        for d in range(1, n // 2 + 1):
            for a in irr[d]:
                for b in monic(n - d):
                    composite.add(mul(a, b))
        irr[n] = [c for c in monic(n) if c not in composite]
    return {n: len(irr[n]) for n in irr}

def gauss(q, n):
    return sum(mobius(d) * q**(n // d) for d in divisors(n)) // n

for p, nmax in [(2, 10), (3, 6)]:
    c = sieve_counts(p, nmax)
    print(f'p = {p}:', c, ' matches Gauss:', all(c[n] == gauss(p, n) for n in c))
for e in [9, 27]:
    f = sympy.factor_list(X**e - X, modulus=3)[1]
    print(f'X^{e} - X over F_3:', sorted(sympy.degree(g, X) for g, _ in f), '(degrees of the factors)')

The sieve and the formula agree. $X^9-X$ has $3$ factors of degree $1$ and $N_3(2)=3$ of degree $2$ ($3+6=9$); $X^{27}-X$ has $3$ linear factors and $N_3(3)=8$ cubics ($3+24=27$), and no quadratics, since $2\nmid3$: $\mathbb F_9\not\subseteq\mathbb F_{27}$.

### P9.4

In [ ]:
H = [[(j >> i) & 1 for j in range(1, 8)] for i in range(3)]       # column j = binary of j
def syndrome(y):
    return sum(((sum(H[i][j] * y[j] for j in range(7)) % 2) << i) for i in range(3))
# codewords: the 16 vectors with zero syndrome
code_words = [c for c in itertools.product((0, 1), repeat=7) if syndrome(c) == 0]
print(len(code_words), 'codewords; minimum distance', min(sum(a != b for a, b in zip(c, d)) for c, d in itertools.combinations(code_words, 2)))
def correct(y):
    s = syndrome(y)
    y = list(y)
    if s:
        y[s - 1] ^= 1         # the syndrome is the position of the error, in binary
    return tuple(y)
ok1 = all(correct(tuple(b ^ (k == j) for k, b in enumerate(c))) == c for c in code_words for j in range(7))
two = [(c, i, j) for c in code_words for i, j in itertools.combinations(range(7), 2)]
detected2 = all(syndrome(tuple(b ^ (k in (i, j)) for k, b in enumerate(c))) != 0 for c, i, j in two)
fixed2 = sum(correct(tuple(b ^ (k in (i, j)) for k, b in enumerate(c))) == c for c, i, j in two)
print('every single error corrected:', ok1, '| double errors detected:', detected2, '| double errors corrected:', fixed2, 'of', len(two))

The $16$ codewords are pairwise at distance $\ge3$, so one error is correctable. The syndrome of a single error in position $j$ is the $j$-th column of $H$, i.e.\ $j$ in binary: the error *tells you where it is*. Two errors give the sum of two different columns, non-zero (so they are detected), but equal to a third column, so the decoder confidently flips a third bit: $0$ of the $336$ double errors are corrected. Reed–Solomon codes do better because they work with whole bytes and many check symbols.

### P9.5

In [ ]:
def solve_mod(A, b, mul, inv, sub):
    # Gaussian elimination over a field given by its operations; returns one solution or None
    n, m = len(A), len(A[0])
    M = [row[:] + [bi] for row, bi in zip(A, b)]
    piv_cols, r = [], 0
    for c in range(m):
        p = next((i for i in range(r, n) if M[i][c] != 0), None)
        if p is None:
            continue
        M[r], M[p] = M[p], M[r]
        iv = inv(M[r][c]); M[r] = [mul(iv, v) for v in M[r]]
        for i in range(n):
            if i != r and M[i][c] != 0:
                f = M[i][c]; M[i] = [sub(v, mul(f, w)) for v, w in zip(M[i], M[r])]
        piv_cols.append(c); r += 1
    if any(all(v == 0 for v in row[:-1]) and row[-1] != 0 for row in M):
        return None
    x = [0] * m
    for i, c in enumerate(piv_cols):
        x[c] = M[i][-1]
    return x

class Field:
    def __init__(self, add, sub, mul, inv): self.add, self.sub, self.mul, self.inv = add, sub, mul, inv
    def pw(self, a, k):
        r = 1
        for _ in range(k): r = self.mul(r, a)
        return r
    def ev(self, P, a):          # Horner, P = coefficients from the constant term up
        r = 0
        for c in reversed(P): r = self.add(self.mul(r, a), c)
        return r
    def divide(self, A, B):      # polynomial division, returns quotient and remainder
        A = A[:]; q = [0] * max(len(A) - len(B) + 1, 1)
        ib = self.inv(B[-1])
        for k in range(len(A) - len(B), -1, -1):
            c = self.mul(A[k + len(B) - 1], ib); q[k] = c
            for i, bi in enumerate(B): A[k + i] = self.sub(A[k + i], self.mul(c, bi))
        return q, A[:len(B) - 1]

GF256 = Field(lambda a, b: a ^ b, lambda a, b: a ^ b, gmul_fast, ginv)

def bw_decode(F, pts, y, k):
    n = len(pts); t = (n - k) // 2
    # unknowns: e_0..e_{t-1}, q_0..q_{k+t-1};  Q(a) - y E(a) = 0 with E monic of degree t
    A, b = [], []
    for a, yi in zip(pts, y):
        row = [F.sub(0, F.mul(yi, F.pw(a, j))) for j in range(t)] + [F.pw(a, j) for j in range(k + t)]
        A.append(row); b.append(F.mul(yi, F.pw(a, t)))
    sol = solve_mod(A, b, F.mul, F.inv, F.sub)
    if sol is None:
        return None
    E = sol[:t] + [1]; Q = sol[t:]
    f, r = F.divide(Q, E)
    return f[:k] if all(v == 0 for v in r) else None

k, n = 20, 30
msg = list(b'Finite fields rock!!')
pts = list(range(1, n + 1))
cw = [GF256.ev(msg, a) for a in pts]
for errors in [5, 6]:
    y = cw[:]
    for i in random.sample(range(n), errors):
        y[i] ^= random.randrange(1, 256)
    out = bw_decode(GF256, pts, y, k)
    print(f'{errors} errors ->', bytes(out) if out else 'decoding failed', '| correct:', out == msg)

p = 7
GF7 = Field(lambda a, b: (a + b) % p, lambda a, b: (a - b) % p, lambda a, b: a * b % p, lambda a: pow(a, p - 2, p))
print('Exercise 9.7: message f =', bw_decode(GF7, [1, 2, 3, 4, 5, 6], [5, 3, 2, 4, 0, 1], 2), '(coefficients from the constant term)')

With $5$ errors (the guaranteed capacity $t=\lfloor(30-20)/2\rfloor=5$) the message comes back intact. With $6$ errors the theorem promises nothing: here the linear system either has no solution, or a solution whose $Q$ is not divisible by $E$, and the decoder reports a failure rather than a wrong message, which is what happens most of the time (a wrong but confident decoding needs the received word to fall within distance $5$ of another codeword, which is rare). Exercise 9.7 gives $f=3+2X$.

Real QR codes use exactly this structure: blocks of up to $255$ bytes over $\mathbb F_{256}$ (with the polynomial $X^8+X^4+X^3+X^2+1$ instead of the AES one), and faster decoders (Berlekamp–Massey and Forney), which also exploit *erasures*: a byte known to be lost (unreadable) costs one check symbol instead of two.